# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mubashir-dev751/starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Random Forest Feature Importance for "Health Score"

The Paper's Claim: In the ML Appendix, a Random Forest model predicts "Health Score," identifying Average Position (43%) and Impressions (32%) as the most important features.

My Methodology Question: The methodology section defines the "Health Score" label as a composite metric calculated directly by adding points for Impressions (30 pts), Position (30 pts), CTR (20 pts), and Scroll Depth (20 pts). The paper acknowledges this overlap. If the target variable is literally a mathematical formula built from the input features, doesn't a feature importance ranking merely reverse-engineer the scoring formula rather than discover a genuine predictive relationship? What is the value of training a model to predict a label using the exact components that define the label?

Finding 2: Logistic Regression Predicting "Growth"

The Paper's Claim: A logistic regression model (71% holdout accuracy) identifies features that separate growing pages from declining pages. It lists "Impressions" and "Days Visible" as top positive coefficients.

My Methodology Question: The paper defines "Trend Direction" (growth vs. decline) based on the "30d-vs-prev-30d impression change". If the model is classifying growth based on an impression-derived label, how were the "Impressions" and "Days Visible" features time-boxed to prevent target leakage? If the features contain impression data from the same 30-day window used to calculate the label, the model is observing the answer during training. Does the validation design strictly separate the feature timeline from the label timeline?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

url = 'https://raw.githubusercontent.com/mubashir-dev751/starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
df['has_no_position_data'] = (df['avg_position'] == 0).astype(int)
df['clean_avg_position'] = np.where(df['avg_position'] == 0, np.nan, df['avg_position'])
df['has_missing_word_count'] = df['word_count'].isna().astype(int)

numeric_features = ['impressions_90d', 'clicks_90d', 'ctr', 'clean_avg_position',
                    'has_no_position_data', 'days_since_last_update', 'has_missing_word_count',
                    'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
categorical_features = ['content_type']
features = numeric_features + categorical_features
target = 'is_declining_label'

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
model = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
])

X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(df[features], df[target], test_size=0.2, random_state=42)
model.fit(X_train_rnd, y_train_rnd)
rnd_auc = roc_auc_score(y_test_rnd, model.predict_proba(X_test_rnd)[:, 1])

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
X_train_grp, X_test_grp = df.iloc[train_idx][features], df.iloc[test_idx][features]
y_train_grp, y_test_grp = df.iloc[train_idx][target], df.iloc[test_idx][target]

model.fit(X_train_grp, y_train_grp)
grp_auc = roc_auc_score(y_test_grp, model.predict_proba(X_test_grp)[:, 1])

print(f"Base Rate (Test): {y_test_grp.mean():.3f}")
print(f"Dishonest Random Split ROC-AUC: {rnd_auc:.3f} (Inflated by client memorization)")
print(f"Honest Grouped Split ROC-AUC:   {grp_auc:.3f} (True generalization)")

Base Rate (Test): 0.511
Dishonest Random Split ROC-AUC: 0.738 (Inflated by client memorization)
Honest Grouped Split ROC-AUC:   0.601 (True generalization)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Leakage Audit Findings:

Overlapping Time Windows: The features impressions_90d and clicks_90d span a trailing 90-day window. If the is_declining_label (trend) is calculated using recent data (e.g., comparing the last 30 days to the prior period), the 90-day totals inherently contain data from after the decline started. The model is partially observing the outcome during training. In a production environment, we must strictly use metrics from before the label window begins.

Missingness Handled Properly: I successfully avoided the missingness trap. avg_position = 0 was converted to NaN and flagged with has_no_position_data rather than allowing a 0 to act as a numerical signal of "high rank."

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The model provides directional decision-support for content teams. By analyzing historical signals—such as page staleness and current search positioning—it generates a ranked queue that surfaces content exhibiting characteristics commonly associated with traffic decline in our observed training data."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.